# TP 3 — Maladie cardiaque : réseaux de neurones et comparaison de modèles

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/ai_for_medicine/blob/main/ia-medecine/notebooks/iam-s3-tp.ipynb)

*IA pour la médecine — séance 3.*

## Objectif

Ce dernier TP réunit tout ce qui a été vu aux séances 1 et 2 :

1. appliquer la **démarche de préparation des données** (séance 2) à un nouveau jeu de données ;
2. entraîner et comparer **trois modèles** : régression logistique, forêt aléatoire et
   **réseau de neurones** (perceptron multicouche, *MLP*) ;
3. comprendre comment un réseau de neurones apprend et comment le régler.

### Le jeu de données : maladie cardiaque (Statlog)

270 patients, 13 variables cliniques. La tâche : prédire la **présence ou l'absence
d'une maladie cardiaque**. Certaines variables sont vraiment numériques (âge, pression
artérielle), d'autres sont des entiers qui codent des catégories (type de douleur
thoracique, thalassémie) : c'est fréquent dans les données médicales réelles.

### La démarche

Toujours la même : **explorer → visualiser → préparer → entraîner → évaluer → comparer**.

**Comment travailler** : exécutez les cellules dans l'ordre (Maj + Entrée) et complétez
les cellules marquées `# TODO`.

---
## 1. Préparation de l'environnement

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, recall_score, ConfusionMatrixDisplay,
                             classification_report)

import warnings
warnings.filterwarnings("ignore")  # masque les avertissements de convergence du MLP

---
## 2. Charger et découvrir les données

**13 variables :**

| Variable | Signification |
|---|---|
| `age` | âge (années) |
| `sexe` | 1 = homme, 0 = femme |
| `type_douleur_thoracique` | 1 angor typique, 2 angor atypique, 3 douleur non angineuse, 4 asymptomatique |
| `pression_arterielle_repos` | pression artérielle au repos (mmHg) |
| `cholesterol` | cholestérol sérique (mg/dL) |
| `glycemie_jeun_sup_120` | glycémie à jeun > 120 mg/dL (1 = oui) |
| `ecg_repos` | ECG au repos : 0 normal, 1 anomalie ST-T, 2 hypertrophie ventriculaire gauche |
| `frequence_cardiaque_max` | fréquence cardiaque maximale atteinte à l'effort |
| `angor_effort` | angor provoqué par l'effort (1 = oui) |
| `depression_st` | sous-décalage du segment ST à l'effort (mm) |
| `pente_st` | pente du segment ST à l'effort : 1 ascendante, 2 plate, 3 descendante |
| `nb_vaisseaux_colores` | nombre de gros vaisseaux visibles en fluoroscopie (0 à 3) |
| `thalassemie` | scintigraphie au thallium : 3 normal, 6 défaut fixe, 7 défaut réversible |

**Cible :** `maladie_cardiaque`, 1 = présente, 0 = absente.

*Source : Statlog (Heart), UCI Machine Learning Repository, CC BY 4.0.*

### Exercice 1.1 — Charger les données

Lisez le fichier avec `pd.read_csv(URL)` dans `df`, affichez sa taille et ses
premières lignes.

In [ ]:
URL = "https://raw.githubusercontent.com/racousin/ai_for_medicine/main/ia-medecine/donnees/maladie_cardiaque.csv"
# TODO : df = ...

### Exercice 1.2 — Statistiques et types

Affichez les types (`df.info()`) et les statistiques descriptives (`df.describe()`).

**Question :** quelles variables représentent des catégories, bien qu'elles soient
stockées sous forme de nombres ?

In [ ]:
# TODO

*Votre réponse ici.*

### Exercice 1.3 — Équilibre des classes

Comptez les patients malades et non malades.

**Pourquoi :** comme à la séance 1, pour savoir si l'exactitude est une métrique fiable.

*Indice :* `y.value_counts(normalize=True)`

In [ ]:
y = df["maladie_cardiaque"]
# TODO

---
## 3. Visualiser les données

### Exercice 2.1 — Distributions selon la classe

Tracez la distribution de `age` et de `frequence_cardiaque_max`, colorée par classe.

*Indice :* `sns.histplot(data=df, x="age", hue="maladie_cardiaque", kde=True)`

In [ ]:
# TODO

### Exercice 2.2 — Nuage de points

Tracez `age` en fonction de `frequence_cardiaque_max`, coloré par classe : les deux
variables ensemble séparent-elles les classes ?

*Indice :* `sns.scatterplot(data=df, x="age", y="frequence_cardiaque_max", hue="maladie_cardiaque", alpha=0.6)`

In [ ]:
# TODO

### Exercice 2.3 — Carte des corrélations

Affichez la matrice des corrélations de toutes les variables.

*Indice :* `sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)`

**Question :** quelles variables semblent les plus liées à la maladie ?

In [ ]:
# TODO

*Votre réponse ici.*

---
## 4. Préparer les données

Ce jeu n'a pas de valeurs manquantes et toutes les variables sont déjà numériques :
il suffit de **standardiser**. Pour les arbres, l'échelle ne compte pas ; pour la
régression logistique et surtout pour les réseaux de neurones, elle est importante.

### Exercice 3.1 — Découper et standardiser

Séparez `X` (toutes les colonnes sauf la cible) et `y`, découpez 80 % / 20 %
(`random_state=42`), puis standardisez avec un `StandardScaler` **ajusté sur
l'entraînement seulement**.

In [ ]:
# TODO : X = ..., y = ...
# TODO : X_train, X_test, y_train, y_test = train_test_split(...)
# TODO : scaler = StandardScaler() ; X_train_scaled = ... ; X_test_scaled = ...

---
## 5. Modèle de référence : la régression logistique

On commence toujours par un modèle de référence simple.

### Exercice 4.1 — Régression logistique

Entraînez `LogisticRegression(max_iter=5000, random_state=42)` et affichez
l'exactitude, le rapport de classification et la matrice de confusion.

In [ ]:
# TODO : lr_model = ... ; lr_model.fit(...)
# TODO : y_pred_lr = ...

---
## 6. Forêt aléatoire

### Exercice 5.1 — Forêt aléatoire

Entraînez `RandomForestClassifier(n_estimators=100, random_state=42)` et comparez-la
à la régression logistique. Calculez aussi son exactitude **sur l'entraînement** :
que constatez-vous ?

In [ ]:
# TODO : rf_model = ... ; y_pred_rf = ...

*Votre réponse ici.*

---
## 7. Réseau de neurones (`MLPClassifier`)

Un **réseau de neurones** transforme l'entrée couche après couche. Chaque neurone
calcule une somme pondérée de ses entrées plus un biais, puis applique une
**fonction d'activation** non linéaire (ReLU par défaut). Les poids, comme les
coefficients θ de la régression logistique, sont **appris** par descente de gradient.

| | Régression logistique | Réseau de neurones (MLP) |
|---|---|---|
| **Couches** | une (entrée → sortie) | plusieurs (entrée → cachées → sortie) |
| **Frontière de décision** | linéaire | non linéaire |
| **Paramètres** | peu (nb de variables + 1) | beaucoup (selon l'architecture) |
| **Entraînement** | rapide, un seul minimum | plus lent, plusieurs minimums possibles |
| **Standardisation** | importante | **indispensable** |

Hyperparamètres principaux de `MLPClassifier` (*Multi-Layer Perceptron*) :

- `hidden_layer_sizes` : `(100,)` = une couche cachée de 100 neurones, `(64, 32)` = deux couches ;
- `activation` : `"relu"` (défaut), `"tanh"`, `"logistic"` ;
- `learning_rate_init` : taux d'apprentissage initial (0,001 par défaut) ;
- `max_iter` : nombre maximal d'époques (passages sur les données).

### Exercice 6.1 — Un premier réseau

Entraînez `MLPClassifier(max_iter=1000, random_state=42)` et comparez-le aux deux
modèles précédents.

In [ ]:
# TODO : mlp_model = ... ; y_pred_mlp = ...

### Exercice 6.2 — La courbe de perte

Tracez la perte d'entraînement à chaque époque : elle dit si le réseau a convergé
(perte qui diminue puis se stabilise).

*Indice :* `plt.plot(mlp_model.loss_curve_)`, et `mlp_model.n_iter_` donne le nombre d'époques.

In [ ]:
# TODO

### Exercice 6.3 — L'arrêt précoce

Entraîné trop longtemps, un réseau sur-apprend : la perte d'entraînement continue de
baisser alors que le modèle se dégrade sur de nouveaux patients. **L'arrêt précoce**
met de côté une partie de l'entraînement comme validation et arrête l'apprentissage
quand le score de validation ne s'améliore plus :

- `early_stopping=True` active l'arrêt précoce ;
- `validation_fraction=0.1` : 10 % de l'entraînement sert de validation ;
- `n_iter_no_change=10` : arrêt après 10 époques sans amélioration.

Entraînez `mlp_es` avec l'arrêt précoce, comparez son nombre d'époques, son
exactitude et sa courbe de perte à celles de `mlp_model`. `mlp_es.validation_scores_`
contient l'exactitude de validation à chaque époque.

In [ ]:
# TODO : mlp_es = MLPClassifier(max_iter=1000, random_state=42, early_stopping=True, validation_fraction=0.1)

*Votre réponse ici.*

---
## 8. Régler le réseau

L'architecture (nombre et taille des couches cachées) change le résultat :

| Architecture | Description |
|---|---|
| `(50,)` | 1 couche, 50 neurones |
| `(100,)` | 1 couche, 100 neurones (défaut) |
| `(64, 32)` | 2 couches (64 → 32) |
| `(128, 64, 32)` | 3 couches : beaucoup de paramètres, il faut beaucoup de données |

**Règle pratique :** commencer simple, n'ajouter de la complexité que si elle aide.
Plus de couches ne veut pas dire meilleur, surtout avec 270 patients.

### Exercice 7.1 — Comparer des architectures

Pour chaque architecture, entraînez un MLP, affichez l'exactitude de test et tracez
les courbes de perte sur un même graphique.

In [ ]:
architectures = [(50,), (100,), (64, 32), (128, 64, 32)]
for arch in architectures:
    # TODO : model = MLPClassifier(hidden_layer_sizes=arch, max_iter=1000, random_state=42)
    # TODO : entraîner, afficher l'exactitude, plt.plot(model.loss_curve_, label=str(arch))
    pass

### Exercice 7.2 — Recherche sur grille

Avec `GridSearchCV` (validation croisée à 5 plis sur l'entraînement), cherchez la
meilleure combinaison de `hidden_layer_sizes`, `activation` et `learning_rate_init`,
puis évaluez le meilleur modèle sur le test.

```python
param_grid = {
    "hidden_layer_sizes": [(50,), (100,), (64, 32)],
    "activation": ["relu", "tanh"],
    "learning_rate_init": [0.001, 0.01],
}
```

In [ ]:
param_grid = {
    "hidden_layer_sizes": [(50,), (100,), (64, 32)],
    "activation": ["relu", "tanh"],
    "learning_rate_init": [0.001, 0.01],
}
# TODO : grid_search_mlp = GridSearchCV(MLPClassifier(max_iter=1000, random_state=42),
#                                       param_grid, cv=5, scoring="accuracy", n_jobs=-1)
# TODO : grid_search_mlp.fit(...) ; afficher best_params_ et best_score_

---
## 9. Comparaison finale

### Exercice 8.1 — Comparaison par validation croisée

Un seul découpage de 54 patients de test est trop petit pour départager les modèles.
Comparez-les par validation croisée à 5 plis sur **toutes** les données. Pour que la
standardisation soit refaite dans chaque pli (sans fuite d'information du pli de
test), on la met dans un **pipeline** : `make_pipeline(StandardScaler(), modèle)`.

In [ ]:
models = {
    "Régression logistique": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, random_state=42)),
    "Forêt aléatoire": RandomForestClassifier(n_estimators=100, random_state=42),
    # TODO : "Réseau de neurones": make_pipeline(StandardScaler(), grid_search_mlp.best_estimator_),
}
for name, model in models.items():
    # TODO : scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
    # TODO : afficher la moyenne et l'écart type
    pass

### Exercice 8.2 — Matrices de confusion côte à côte

Affichez les matrices de confusion des trois modèles sur le jeu de test.

*Indice :* `fig, axes = plt.subplots(1, 3, figsize=(18, 5))`

**Question :** pour dépister une maladie cardiaque, quelle case de la matrice
regardez-vous en premier ?

In [ ]:
# TODO

*Votre réponse ici.*

---
## 10. Bilan des trois séances

| | Régression logistique | Forêt aléatoire | Réseau de neurones (MLP) |
|---|---|---|---|
| **Type** | linéaire | ensemble d'arbres | couches de neurones |
| **Frontière** | linéaire | non linéaire | non linéaire |
| **Standardisation** | importante | inutile | indispensable |
| **Interprétabilité** | forte (coefficients) | moyenne (importance des variables) | faible (boîte noire) |
| **Quand l'utiliser** | référence, petits jeux | données tabulaires, défaut robuste | motifs complexes, beaucoup de données |

**La démarche (pour tous les modèles)** : explorer, visualiser, préparer, découper,
entraîner (en commençant par une référence), évaluer avec la bonne métrique,
régler par validation croisée, comparer.

> En santé, **le choix de la métrique compte plus que le choix du modèle**. Un
> modèle exact à 95 % mais qui manque des malades est dangereux : un diagnostic
> manqué (faux négatif) coûte en général bien plus qu'une fausse alerte.